# 12 - Final Model Training

Model development and hyperparameter selection are now complete.

The configurations of the six models were chosen using temporal cross-validation on the 2015-2022 training period and then checked on the external 2023 validation set.

At this point no more model choices are made, so we can add the 2023 validation data back to the training data and use all matches from 2015 to 2023 to train the final models.

The final split is therefore:

- **Final training:** 2015-2023
- **Final test:** 2024-2025

The 2024-2025 test set remains completely separate from training and preprocessing.

In this notebook we:
- fit the final preprocessing
- train the six selected models
- generate their final predictions and probabilities
- compare them with the ranking baseline
- save everything needed for the final evaluation

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

import sys
import joblib

# scaling is needed by ANN and k-NN
from sklearn.preprocessing import StandardScaler

# models selected in the previous notebooks
from sklearn.ensemble import (
    RandomForestClassifier,
    AdaBoostClassifier
)

from sklearn.tree import DecisionTreeClassifier

from sklearn.neighbors import (
    KNeighborsClassifier
)

from xgboost import XGBClassifier

from lightgbm import LGBMClassifier

# TensorFlow is used for the neural network
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense,
    Activation
)

from tensorflow.keras import Input
from tensorflow.keras import optimizers

# add the project root so that functions from src/ can be imported
PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(
        str(PROJECT_ROOT)
    )

from src.preprocessing import (
    fit_preprocessor,
    transform_data,
    numeric_features
)

from src.evaluation import (
    evaluate_model
)

In [2]:
# use the same random seeds so that results are as reproducible as possible
np.random.seed(42)
tf.random.set_seed(42)

In [3]:
# folders used to load the data and save the final results/models
PROCESSED_DATA_DIR = Path(
    "../data/processed"
)

RESULTS_DIR = Path(
    "../results"
)

FINAL_RESULTS_DIR = (
    RESULTS_DIR
    / "final"
)

MODELS_DIR = Path(
    "../models"
)

# create the output folders if they do not already exist
FINAL_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

## Load the prepared dataset

We start from the dataset created in the previous notebooks.

It already contains the Training, Validation and Test labels, so we do not create a new split here.

We sort the matches again by date and MatchID to make sure the chronological order is preserved.

In [4]:
# load the complete featured dataset with its split labels
matches = pd.read_csv(
    PROCESSED_DATA_DIR / "feature_split_matches.csv",
    parse_dates=["Date"]
)

# keep matches in chronological order
matches = (
    matches
    .sort_values(["Date", "MatchID"])
    .reset_index(drop=True)
)


print("Dataset shape:", matches.shape)

print()

# check how many observations belong to each original split
print(
    matches["DataSplit"].value_counts()
)

Dataset shape: (26536, 53)

DataSplit
Training      18810
Test           5119
Validation     2607
Name: count, dtype: int64


## Create the final train/test split

During model development, 2023 was kept separate as an external validation set.

Now that all model choices and hyperparameters are fixed, we can include those matches in the final training data.

Therefore:

- the old Training + Validation sets become the final training set
- the original Test set remains untouched

This gives the models as much past information as possible before evaluating them on 2024-2025.

In [5]:
# combine the original training and validation periods
# to obtain the final 2015-2023 training set
final_train_data = (
    matches.loc[
        matches["DataSplit"].isin(
            [
                "Training",
                "Validation"
            ]
        )
    ].copy()
)

# the original test period remains unchanged
test_data = (
    matches.loc[
        matches["DataSplit"] == "Test"
    ].copy()
)

print(
    "Final training matches:",
    len(final_train_data)
)

print(
    "Final test matches:",
    len(test_data)
)

print()

print(
    "Final training period:",
    final_train_data["Date"].min(),
    "-",
    final_train_data["Date"].max()
)

print(
    "Final test period:",
    test_data["Date"].min(),
    "-",
    test_data["Date"].max()
)

Final training matches: 21417
Final test matches: 5119

Final training period: 2015-01-05 00:00:00 - 2023-12-31 00:00:00
Final test period: 2024-01-01 00:00:00 - 2025-11-16 00:00:00


In [6]:
# final training must finish before the test period begins
assert (
    final_train_data["Date"].max() < test_data["Date"].min()
)

# final training should contain only the old Training and Validation sets
assert set(
    final_train_data["DataSplit"].unique()) == {
        "Training",
        "Validation"
    }

# test data must contain only the original Test split
assert set(
    test_data["DataSplit"].unique()) == {
    "Test"
}

print(
    "Final chronological split is correct."
)

Final chronological split is correct.


## Selected model configurations

The hyperparameters below are not chosen in this notebook.

They are the configurations selected during the previous model-selection notebooks using temporal cross-validation.

Keeping them fixed is important because the final test set must not influence model selection.

In [7]:
# configurations selected during the previous tuning stage
selected_configurations = pd.DataFrame(
    [
        {
            "Model":
                "ANN",
            "Configuration":
                "16 neurons, ReLU, learning rate 0.001, 50 epochs"
        },

        {
            "Model":
                "Random Forest",
            "Configuration":
                "200 trees, max depth 10"
        },

        {
            "Model":
                "k-NN",
            "Configuration":
                "151 neighbors, uniform weights"
        },

        {
            "Model":
                "AdaBoost",
            "Configuration":
                "4 max leaf nodes, 100 estimators, learning rate 0.5"
        },

        {
            "Model":
                "XGBoost",
            "Configuration":
                "max depth 2, 200 estimators, learning rate 0.05"
        },

        {
            "Model":
                "LightGBM",
            "Configuration":
                "7 leaves, 200 estimators, learning rate 0.05"
        }
    ]
)


selected_configurations

,Model,Configuration
0,ANN,"16 neurons, ReLU, learning rate 0.001, 50 epochs"
1,Random Forest,"200 trees, max depth 10"
2,k-NN,"151 neighbors, uniform weights"
3,AdaBoost,"4 max leaf nodes, 100 estimators, learning rat..."
4,XGBoost,"max depth 2, 200 estimators, learning rate 0.05"
5,LightGBM,"7 leaves, 200 estimators, learning rate 0.05"


## Final preprocessing

The preprocessing now has to be fitted again because the final training set contains more data than before.

Missing-value replacements and one-hot encoding are therefore learned using all matches from 2015 to 2023.

The 2024-2025 test data is only transformed using these learned values. It is never used to fit the preprocessing.

In [8]:
# learn the final preprocessing rules from 2015-2023 only
(
    final_numeric_medians,
    final_categorical_modes,
    final_one_hot_encoder
) = fit_preprocessor(
    final_train_data
)

# transform final training data
X_final_train = transform_data(
    final_train_data,
    final_numeric_medians,
    final_categorical_modes,
    final_one_hot_encoder
)

# apply exactly the same preprocessing to the test data
X_test = transform_data(
    test_data,
    final_numeric_medians,
    final_categorical_modes,
    final_one_hot_encoder
)

# target values
y_final_train = (
    final_train_data["Player1Won"].to_numpy()
)

y_test = (
    test_data["Player1Won"].to_numpy()
)

In [9]:
# check that training and test have compatible processed data
print(
    "Final training matrix:",
    X_final_train.shape
)

print(
    "Test matrix:",
    X_test.shape
)

# column names and order should also be identical
assert (
    list(X_final_train.columns) == list(X_test.columns)
)

# preprocessing should have removed all missing values
assert (X_final_train.isna().sum().sum()==0)
assert (X_test.isna().sum().sum()==0)

print(
    "Final preprocessing check passed."
)

Final training matrix: (21417, 50)
Test matrix: (5119, 50)
Final preprocessing check passed.


In [10]:
# keep all the fitted preprocessing objects together
final_preprocessing = {
    "numeric_medians": final_numeric_medians,
    "categorical_modes": final_categorical_modes,
    "one_hot_encoder": final_one_hot_encoder
}

# save them so future data can be processed in exactly the same way
joblib.dump(
    final_preprocessing,
    MODELS_DIR / "final_preprocessing.joblib"
)

print("Final preprocessing saved.")

Final preprocessing saved.


### Scaling for ANN and k-NN

ANN and k-NN need scaled numerical features.

k-NN calculates distances directly, while neural-network optimization also works better when numerical inputs are on similar scales.

The tree-based models do not need this scaling because tree splits depend on thresholds rather than distances.

As always, the scaler is fitted only on the final training data and then applied to the test data.

In [11]:
# one final scaler is enough for both ANN and k-NN
final_scaler = StandardScaler()

# keep the original unscaled data for the tree-based models
X_final_train_scaled = (
    X_final_train.copy()
)

X_test_scaled = (
    X_test.copy()
)

# learn mean and standard deviation from final training data
X_final_train_scaled[numeric_features] = (
    final_scaler.fit_transform(
        X_final_train_scaled[numeric_features]
    )
)

# apply the same scaling parameters to test data
X_test_scaled[numeric_features] = (
    final_scaler.transform(
        X_test_scaled[numeric_features]
    )
)

# save the scaler together with the final models
joblib.dump(
    final_scaler,
    MODELS_DIR / "final_scaler.joblib"
)

print("Final scaler saved.")

Final scaler saved.


## Final Artificial Neural Network

We use the same ANN architecture selected during model development.

The network has one hidden layer with 16 neurons and ReLU activation.

The output layer contains one neuron with sigmoid activation because this is a binary classification problem. Its output can therefore be interpreted as the estimated probability that Player 1 wins.

In [12]:
# helper function used to create the ANN architecture
def build_ann(
    input_features,
    hidden_layers,
    hidden_activation,
    learning_rate
):

    model = Sequential()

    # one input value for every processed feature
    model.add(
        Input(shape=(input_features,))
    )

    # create the hidden layers
    for number_of_neurons in hidden_layers:

        model.add(
            Dense(number_of_neurons)
        )

        model.add(
            Activation(hidden_activation)
        )

    # one output neuron because the target is binary
    model.add(
        Dense(1)
    )

    # sigmoid converts the output into a value between 0 and 1
    model.add(
        Activation("sigmoid")
    )

    model.compile(
        # Adam updates the neural-network weights during training
        optimizer=optimizers.Adam(learning_rate=learning_rate),

        # standard loss function for binary classification
        loss="binary_crossentropy",

        metrics=["accuracy"]
    )

    return model

In [13]:
# reset the random seeds before training the final ANN
np.random.seed(42)
tf.random.set_seed(42)

# TensorFlow works efficiently with NumPy float32 arrays
X_ann_train = (
    X_final_train_scaled.to_numpy(dtype=np.float32)
)

X_ann_test = (
    X_test_scaled.to_numpy(dtype=np.float32)
)

# build the configuration selected during ANN tuning
final_ann = build_ann(
    input_features=X_ann_train.shape[1],
    hidden_layers=[16],
    hidden_activation="relu",
    learning_rate=0.001
)

# train on all 2015-2023 observations
final_ann.fit(
    X_ann_train,
    y_final_train,
    epochs=50,
    batch_size=128,
    verbose=1
)

Epoch 1/50
168/168 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6022 - loss: 0.6674
Epoch 2/50
168/168 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.6492 - loss: 0.6202
Epoch 3/50
168/168 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6551 - loss: 0.6139
Epoch 4/50
168/168 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6591 - loss: 0.6112
Epoch 5/50
168/168 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6608 - loss: 0.6093
Epoch 6/50
168/168 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6629 - loss: 0.6080
Epoch 7/50
168/168 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.6642 - loss: 0.6069
Epoch 8/50
168/168 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6640 - loss: 0.6061
Epoch 9/50
168/168 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.6647 - loss: 0.6054
Epoch 10/50
168/168 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.6648 - loss: 0.6049
Epoch 11/50
168/168 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.6662 - loss: 0.6043
Epoch 12/50
168/168 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step

In [14]:
# predicted probability that Player 1 wins
ann_test_probabilities = (
    final_ann.predict(
        X_ann_test,
        verbose=0
    ).reshape(-1)
)

# convert probabilities into class predictions using 0.5 as threshold
ann_test_predictions = (
    ann_test_probabilities>=0.5
).astype(int)

# calculate the final test metrics
ann_test_result = (
    evaluate_model(
        y_true=y_test,
        y_pred=ann_test_predictions,
        model_name="ANN",
        y_probability=ann_test_probabilities
    )
)

ann_test_result

,Model,Accuracy,BalancedAccuracy,Precision,Recall,F1,ROC-AUC,LogLoss
0,ANN,0.641727,0.641807,0.648873,0.634325,0.641517,0.706608,0.622347


In [15]:
# save the final trained neural network
final_ann.save(
    MODELS_DIR / "final_ann.keras"
)

print("Final ANN saved.")

Final ANN saved.


## Final traditional and ensemble models

We now train the other five selected models.

The hyperparameters are exactly the ones selected during their individual model-selection notebooks.

Random Forest, AdaBoost, XGBoost and LightGBM use the normal processed data, while k-NN uses the scaled version.

In [16]:
# final Random Forest configuration selected previously
final_rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=1,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

# train on all 2015-2023 matches
final_rf.fit(
    X_final_train,
    y_final_train
)

# final class predictions
rf_test_predictions = (
    final_rf.predict(X_test)
)

# probability that Player 1 wins
rf_test_probabilities = (
    final_rf.predict_proba(X_test)[:, 1]
)

# evaluate on the untouched test set
rf_test_result = (
    evaluate_model(
        y_true=y_test,
        y_pred=rf_test_predictions,
        model_name="Random Forest",
        y_probability=rf_test_probabilities
    )
)

rf_test_result

# save the trained model
joblib.dump(
    final_rf,
    MODELS_DIR / "final_rf.joblib"
)

print("Final Random Forest saved.")

Final Random Forest saved.


In [17]:
# final k-NN configuration selected by temporal CV
final_knn = KNeighborsClassifier(
    n_neighbors=151,
    weights="uniform",
    n_jobs=-1
)

# k-NN uses the scaled dataset because its predictions depend on distances
final_knn.fit(
    X_final_train_scaled,
    y_final_train
)

knn_test_predictions = (
    final_knn.predict(X_test_scaled)
)

knn_test_probabilities = (
    final_knn.predict_proba(X_test_scaled)[:, 1]
)

knn_test_result = (
    evaluate_model(
        y_true=y_test,
        y_pred=knn_test_predictions,
        model_name="k-NN",
        y_probability=knn_test_probabilities
    )
)

knn_test_result

joblib.dump(
    final_knn,
    MODELS_DIR / "final_knn.joblib"
)

print("Final k-NN saved.")

Final k-NN saved.


In [18]:
# weak learner used inside AdaBoost
# each small tree is intentionally kept simple
final_weak_tree = (
    DecisionTreeClassifier(
        max_leaf_nodes=4,
        random_state=42
    )
)

# AdaBoost combines many weak trees sequentially
final_adaboost = (
    AdaBoostClassifier(
        estimator=final_weak_tree,
        n_estimators=100,
        learning_rate=0.5,
        random_state=42
    )
)

# train the ensemble
final_adaboost.fit(
    X_final_train,
    y_final_train
)

adaboost_test_predictions = (
    final_adaboost.predict(X_test)
)


adaboost_test_probabilities = (
    final_adaboost.predict_proba(X_test)[:, 1]
)

adaboost_test_result = (
    evaluate_model(
        y_true=y_test,
        y_pred=adaboost_test_predictions,
        model_name="AdaBoost",
        y_probability=adaboost_test_probabilities
    )
)

adaboost_test_result

joblib.dump(
    final_adaboost,
    MODELS_DIR / "final_adaboost.joblib"
)


print("Final AdaBoost saved.")

Final AdaBoost saved.


In [19]:
# final XGBoost configuration selected during tuning
final_xgboost = XGBClassifier(
    n_estimators=200,
    max_depth=2,
    learning_rate=0.05,
    
    # binary logistic classification produces probabilities
    objective="binary:logistic",
    
    # metric used internally by XGBoost
    eval_metric="logloss",
    
    # feature importance will later be based on improvement in the objective
    importance_type="gain",
    random_state=42,
    n_jobs=-1
)

final_xgboost.fit(
    X_final_train,
    y_final_train
)

xgboost_test_predictions = (
    final_xgboost.predict(X_test)
)

xgboost_test_probabilities = (
    final_xgboost.predict_proba(X_test)[:, 1]
)

xgboost_test_result = (
    evaluate_model(
        y_true=y_test,
        y_pred=xgboost_test_predictions,
        model_name="XGBoost",
        y_probability=xgboost_test_probabilities
    )
)


xgboost_test_result

joblib.dump(
    final_xgboost,
    MODELS_DIR / "final_xgboost.joblib"
)


print("Final XGBoost saved.")

Final XGBoost saved.


In [20]:
# final LightGBM configuration selected during tuning
final_lightgbm = LGBMClassifier(
    num_leaves=7,
    n_estimators=200,
    learning_rate=0.05,
    objective="binary",
    random_state=42,
    n_jobs=-1,
    
    # avoid printing LightGBM training messages
    verbosity=-1
)


final_lightgbm.fit(
    X_final_train,
    y_final_train
)

lightgbm_test_predictions = (
    final_lightgbm.predict(X_test)
)


lightgbm_test_probabilities = (
    final_lightgbm.predict_proba(X_test)[:, 1]
)


lightgbm_test_result = (
    evaluate_model(
        y_true=y_test,
        y_pred=lightgbm_test_predictions,
        model_name="LightGBM",
        y_probability=lightgbm_test_probabilities
    )
)

lightgbm_test_result

joblib.dump(
    final_lightgbm,
    MODELS_DIR / "final_lightgbm.joblib"
)


print("Final LightGBM saved.")

Final LightGBM saved.


## Compare the six final models

All six models have now produced predictions on exactly the same 2024-2025 test matches.

We combine their metrics into one table and order them mainly by accuracy.

ROC-AUC is used as a second sorting criterion if two models have the same accuracy.

In [21]:
# put the final metrics of all six models into one table
final_model_results = pd.concat(
    [
        ann_test_result,
        rf_test_result,
        knn_test_result,
        adaboost_test_result,
        xgboost_test_result,
        lightgbm_test_result
    ],

    ignore_index=True
)

# show the strongest final results first
final_model_results = (
    final_model_results
    .sort_values(
        by=["Accuracy","ROC-AUC"],
        ascending=[False, False]
    ).reset_index(drop=True)
)


final_model_results

,Model,Accuracy,BalancedAccuracy,Precision,Recall,F1,ROC-AUC,LogLoss
0,XGBoost,0.650908,0.651042,0.659744,0.638578,0.648988,0.711920,0.617871
1,AdaBoost,0.647978,0.648009,0.653741,0.645149,0.649416,0.710363,0.624850
2,LightGBM,0.647587,0.647811,0.659082,0.626981,0.642631,0.709218,0.620534
3,Random Forest,0.646611,0.646841,0.658259,0.625435,0.641427,0.709821,0.618960
4,k-NN,0.642313,0.642698,0.658557,0.606881,0.631664,0.699133,0.628915
5,ANN,0.641727,0.641807,0.648873,0.634325,0.641517,0.706608,0.622347


XGBoost obtains the highest final test accuracy at approximately **65.1%**, although AdaBoost, LightGBM and Random Forest are very close.

The small differences between the ensemble models suggest that none of them has a very large advantage over the others on this dataset.

In [22]:
# save the final metrics for the evaluation notebook
final_model_results.to_csv(
    FINAL_RESULTS_DIR / "final_model_results.csv",
    index=False
)

print("Final model metrics saved.")

Final model metrics saved.


## Better-ranked-player baseline

We also evaluate the simple ranking baseline on the final test set.

The rule predicts that whichever player has the better ATP ranking will win.

This gives an important reference point because a machine-learning model should ideally provide useful information beyond ranking alone.

In [23]:
# ranking is required for every test match to apply this baseline
assert (
    test_data[
        [
            "Player1Rank",
            "Player2Rank"
        ]
    ].isna().sum().sum()==0
)

# smaller ATP rank numbers are better
# predict Player 1 when Player 1 has the lower ranking number
ranking_baseline_predictions = (
    test_data["Player1Rank"] < test_data["Player2Rank"]
).astype(int).to_numpy()

ranking_baseline_result = (
    evaluate_model(
        y_true=y_test,
        y_pred=ranking_baseline_predictions,
        model_name="Better-ranked player"
    )
)

ranking_baseline_result

,Model,Accuracy,BalancedAccuracy,Precision,Recall,F1
0,Better-ranked player,0.642508,0.642467,0.646308,0.646308,0.646308


In [24]:
# combine models and ranking baseline in one comparison table
final_test_comparison = (
    pd.concat(
        [
            final_model_results,
            ranking_baseline_result
        ],
        ignore_index=True
    )
)

final_test_comparison = (
    final_test_comparison.sort_values(
        "Accuracy",
        ascending=False
    ).reset_index(drop=True)
)

final_test_comparison

# save it for the detailed final evaluation
final_test_comparison.to_csv(
    FINAL_RESULTS_DIR / "final_test_comparison.csv",
    index=False
)

XGBoost reaches approximately **65.1% accuracy**, compared with approximately **64.3% for the ranking baseline**.

This means that the best machine-learning model improves on the simple ranking rule, although the improvement is relatively small.

## Create the final prediction dataset

The previous tables contain only overall metrics.

For the detailed error analysis we also need the prediction made by every model for every individual test match.

We therefore create one common dataset containing the main match information together with each model's:

- predicted winner
- probability for Player 1
- whether the prediction was correct
- confidence

In [25]:
# keep the match information that will be useful during error analysis
analysis_columns = [
    "MatchID",
    "Date",
    "Tournament",
    "Surface",
    "Round",

    "Player1",
    "Player2",

    "Player1Rank",
    "Player2Rank",

    "RankDifference",
    "PointsDifference",

    "WinRateDifference",
    "Recent5WinRateDifference",
    "SurfaceWinRateDifference",

    "H2HMatchesBefore",

    "Player1Won"
]

final_test_predictions = (
    test_data[analysis_columns].copy()
)

In [26]:
# add ANN outputs for every test match
final_test_predictions["ANNPrediction"] = ann_test_predictions

# probability assigned to Player 1 winning
final_test_predictions["ANNProbability"] = ann_test_probabilities

# True when the prediction matches the real winner
final_test_predictions["ANNCorrect"] = (
    ann_test_predictions == y_test
)

# confidence is the probability assigned to whichever class was predicted
final_test_predictions["ANNConfidence"] = np.maximum(
    ann_test_probabilities,
    1 - ann_test_probabilities
)

In [27]:
# add Random Forest predictions, probabilities, correctness and confidence
final_test_predictions["RFPrediction"] = rf_test_predictions

final_test_predictions["RFProbability"] = rf_test_probabilities

final_test_predictions["RFCorrect"] = (
    rf_test_predictions == y_test
)

final_test_predictions[
    "RFConfidence"
] = np.maximum(
    rf_test_probabilities,
    1 - rf_test_probabilities
)

In [28]:
# add k-NN predictions, probabilities, correctness and confidence
final_test_predictions[
    "kNNPrediction"
] = knn_test_predictions


final_test_predictions[
    "kNNProbability"
] = knn_test_probabilities


final_test_predictions[
    "kNNCorrect"
] = ( knn_test_predictions == y_test)


final_test_predictions[
    "kNNConfidence"
] = np.maximum(
    knn_test_probabilities,
    1 - knn_test_probabilities
)

In [29]:
# add AdaBoost predictions, probabilities, correctness and confidence
final_test_predictions[
    "AdaBoostPrediction"
] = adaboost_test_predictions


final_test_predictions[
    "AdaBoostProbability"
] = adaboost_test_probabilities


final_test_predictions[
    "AdaBoostCorrect"
] = (
    adaboost_test_predictions == y_test
)


final_test_predictions[
    "AdaBoostConfidence"
] = np.maximum(
    adaboost_test_probabilities,
    1 - adaboost_test_probabilities
)

In [30]:
# add XGBoost predictions, probabilities, correctness and confidence
final_test_predictions[
    "XGBoostPrediction"
] = xgboost_test_predictions


final_test_predictions[
    "XGBoostProbability"
] = xgboost_test_probabilities


final_test_predictions[
    "XGBoostCorrect"
] = (
    xgboost_test_predictions == y_test
)


final_test_predictions[
    "XGBoostConfidence"
] = np.maximum(
    xgboost_test_probabilities,
    1 - xgboost_test_probabilities
)

In [31]:
# add LightGBM predictions, probabilities, correctness and confidence
final_test_predictions[
    "LightGBMPrediction"
] = lightgbm_test_predictions


final_test_predictions[
    "LightGBMProbability"
] = lightgbm_test_probabilities


final_test_predictions[
    "LightGBMCorrect"
] = (
    lightgbm_test_predictions == y_test
)


final_test_predictions[
    "LightGBMConfidence"
] = np.maximum(
    lightgbm_test_probabilities,
    1 - lightgbm_test_probabilities
)

In [32]:
# also add the ranking-baseline prediction for comparison
final_test_predictions[
    "RankingBaselinePrediction"
] = (
    ranking_baseline_predictions
)


final_test_predictions[
    "RankingBaselineCorrect"
] = (
    ranking_baseline_predictions == y_test
)

In [33]:
# save one complete file with predictions from every final model
final_test_predictions.to_csv(
    FINAL_RESULTS_DIR
    / "final_test_predictions.csv",
    index=False
)

print("Final test predictions saved.")

Final test predictions saved.


## Final checks

Before finishing the notebook, we verify that the expected output files were created and that the prediction dataset has the correct number of observations and no missing model probabilities.

In [34]:
# show all CSV files created in the final-results folder
# glob("*.csv") searches the folder for every filename ending in .csv.
print( "Final result files:")

for file_path in sorted(
    FINAL_RESULTS_DIR.glob("*.csv")
):  
    print("-", file_path.name)

Final result files:
- bookmaker_benchmark.csv
- correct_vs_wrong_feature_distribution.csv
- final_model_results.csv
- final_test_comparison.csv
- final_test_predictions.csv


In [35]:
# there must be exactly one prediction row for every test match
assert len(final_test_predictions) == len(test_data)

# every model must have produced a probability for every test match
assert (
    final_test_predictions[
        [
            "ANNProbability",
            "RFProbability",
            "kNNProbability",
            "AdaBoostProbability",
            "XGBoostProbability",
            "LightGBMProbability"
        ]
    ].isna().sum().sum()==0
)

print("Final prediction checks passed.")

# inspect the first few rows manually
final_test_predictions.head()

Final prediction checks passed.


,MatchID,Date,Tournament,Surface,Round,Player1,Player2,Player1Rank,Player2Rank,RankDifference,...,XGBoostPrediction,XGBoostProbability,XGBoostCorrect,XGBoostConfidence,LightGBMPrediction,LightGBMProbability,LightGBMCorrect,LightGBMConfidence,RankingBaselinePrediction,RankingBaselineCorrect
21417,21417,2024-01-01,Brisbane International,Hard,1st Round,Murray A.,Dimitrov G.,42.0,14.0,-28.0,...,0,0.383699,True,0.616301,0,0.316778,True,0.683222,0,True
21418,21418,2024-01-01,Brisbane International,Hard,1st Round,Rune H.,Purcell M.,8.0,45.0,37.0,...,1,0.778013,True,0.778013,1,0.775602,True,0.775602,1,True
21419,21419,2024-01-01,Brisbane International,Hard,1st Round,Shelton B.,Safiullin R.,17.0,39.0,22.0,...,1,0.634187,False,0.634187,1,0.643646,False,0.643646,1,False
21420,21420,2024-01-01,Hong Kong Tennis Open,Hard,1st Round,Borges N.,Kotov P.,66.0,67.0,1.0,...,0,0.440316,True,0.559684,0,0.438579,True,0.561421,1,False
21421,21421,2024-01-01,Hong Kong Tennis Open,Hard,1st Round,Bonzi B.,Ruusuvuori E.,73.0,69.0,-4.0,...,0,0.485134,True,0.514866,1,0.525389,False,0.525389,0,True


In [36]:
# final overview of the training and test periods and model results
print("=" * 70)

print("Final model training complete")

print("=" * 70)


print("\nFinal training period:")
print(
    final_train_data["Date"].min(),
    "-",
    final_train_data["Date"].max()
)


print("\nFinal test period:")
print(
    test_data["Date"].min(),
    "-",
    test_data["Date"].max()
)


print("\nFinal model results:")
display(final_model_results)


print("\nModels + baseline:")
display(final_test_comparison)

Final model training complete

Final training period:
2015-01-05 00:00:00 - 2023-12-31 00:00:00

Final test period:
2024-01-01 00:00:00 - 2025-11-16 00:00:00

Final model results:


,Model,Accuracy,BalancedAccuracy,Precision,Recall,F1,ROC-AUC,LogLoss
0,XGBoost,0.650908,0.651042,0.659744,0.638578,0.648988,0.711920,0.617871
1,AdaBoost,0.647978,0.648009,0.653741,0.645149,0.649416,0.710363,0.624850
2,LightGBM,0.647587,0.647811,0.659082,0.626981,0.642631,0.709218,0.620534
3,Random Forest,0.646611,0.646841,0.658259,0.625435,0.641427,0.709821,0.618960
4,k-NN,0.642313,0.642698,0.658557,0.606881,0.631664,0.699133,0.628915
5,ANN,0.641727,0.641807,0.648873,0.634325,0.641517,0.706608,0.622347



Models + baseline:


,Model,Accuracy,BalancedAccuracy,Precision,Recall,F1,ROC-AUC,LogLoss
0,XGBoost,0.650908,0.651042,0.659744,0.638578,0.648988,0.711920,0.617871
1,AdaBoost,0.647978,0.648009,0.653741,0.645149,0.649416,0.710363,0.624850
2,LightGBM,0.647587,0.647811,0.659082,0.626981,0.642631,0.709218,0.620534
3,Random Forest,0.646611,0.646841,0.658259,0.625435,0.641427,0.709821,0.618960
4,Better-ranked player,0.642508,0.642467,0.646308,0.646308,0.646308,NaN,NaN
5,k-NN,0.642313,0.642698,0.658557,0.606881,0.631664,0.699133,0.628915
6,ANN,0.641727,0.641807,0.648873,0.634325,0.641517,0.706608,0.622347


## Final result

All six selected models were successfully retrained using the complete 2015-2023 period and evaluated on the untouched 2024-2025 test set.

XGBoost achieved the highest final accuracy at approximately **65.1%**, followed closely by AdaBoost, LightGBM and Random Forest.

The better-ranked-player baseline achieved approximately **64.3%**, showing that ATP ranking alone is already a strong predictor and that the improvement obtained by the machine-learning models is relatively small.

The detailed analysis of these final predictions is carried out in the next notebook.